In [3]:
import scanpy as sc

In [4]:
from matplotlib import colormaps

from colorspacious import cspace_converter

import matplotlib.pyplot as plt
import numpy as np

import matplotlib as mpl

col_map_red = mpl.colormaps['Reds'].resampled(256)

In [5]:
# ran out of memory when merging  all 4
# ## takes like 10 min to merge everything

In [6]:
# using 700gb of memory in condo

In [7]:
# using fibrosis stage 2 sample in place of metald

In [8]:
import os
import scanpy as sc

def process_sample(
    samples, 
    color_map, 
    gene_markers, 
    idir, 
    filt_h5ad_dir, 
    odir
):
    """
    Processes spatial transcriptomics data for a list of samples.

    Parameters:
        samples (list): List of sample names.
        color_map (dict): Dictionary mapping cell types to colors.
        gene_markers (list): List of gene markers for heatmap generation.
        idir (str): Input directory containing spatial data.
        odir (str): Output directory for saving results.
        create_tangram_scores_df (func): Function to generate tangram scores DataFrame.
    """
    os.makedirs(odir, exist_ok = True)
    for sample_nm in samples:
        print(f"Processing {sample_nm}...")
        
        # Define input and output file paths
        # idir for imputed
        # idir for spatial with predicted celltype
        sample_dir = os.path.join(idir, sample_nm)
        adge_file = os.path.join(sample_dir, "merge", "adge_merge.h5ad")

        filt_h5ad_sample_dir = os.path.join(filt_h5ad_dir, sample_nm)
        ad_sp_file = os.path.join(filt_h5ad_sample_dir, "adata_0.5_tangram_filt.h5ad")
        patdir = os.path.join(odir, sample_nm)
        os.makedirs(patdir, exist_ok=True)

        # Read data
        adge = sc.read_h5ad(adge_file)
        adata_pred_filt = sc.read_h5ad(ad_sp_file)
        # Subsetting adge based on cells in adata_pred (should not have undetermined cells)
        subset_cells = adata_pred_filt.obs_names  # Extract cell names from the subset object
        # Subset the original AnnData object
        adge_filt = adge[adge.obs_names.isin(subset_cells)].copy()
        
        # Subset and sync spatial data
        adge_filt.obs['pred_cell_types'] = adata_pred_filt.obs['pred_cell_types']
        adge_filt.obsm['spatial'] = adata_pred_filt.obsm['spatial']
        adge_filt.uns['spatial'] = adata_pred_filt.uns['spatial']

        # Plot spatial data by cell type
        #plot_spatial_by_celltype(adata_pred_filt, outdir=patdir, sample_nm=sample_nm, palette_dict=color_map)

        # Create expression heatmaps
        create_expression_heatmaps(
            raw_adata=adata_pred_filt,
            imputed_adata=adge_filt,
            markers=gene_markers,
            groupby="pred_cell_types",
            outdir=patdir,
            sample_name=sample_nm
        )

        # Plot n counts distribution
        #plot_n_counts_distribution(
        #    adata=adata_pred_filt,
        #    output_dir=patdir,
        #    sample_name=sample_nm
        #)


In [9]:
idir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/deep_seq/'

In [10]:
filt_h5ad_dir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/misc/tangram_scores_dist2/'

In [13]:
odir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/misc/qc_plots_comb/'

In [6]:
# QY_2673_1_2	HL230324	metald,   80G
# QY_2665_1_2	HL160029	Normal,  90G
# QY_2480_Liver_MASH	HL2201019	MASH HL20221019_broad_no_mast_filt/merge/adge_merge.h5ad (88GB)

# HL170058_filt MASH QY_2666

In [7]:
# READ IN masl

In [11]:
#sample_nm = 'HL2201019

sample_nm = 'HL230324_filt'

In [14]:
# idir for imputed
# idir for spatial with predicted celltype
sample_dir = os.path.join(idir, sample_nm)
adge_file = os.path.join(sample_dir, "merge", "adge_merge.h5ad")

filt_h5ad_sample_dir = os.path.join(filt_h5ad_dir, sample_nm)
ad_sp_file = os.path.join(filt_h5ad_sample_dir, "adata_0.5_tangram_filt.h5ad")
patdir = os.path.join(odir, sample_nm)
os.makedirs(patdir, exist_ok=True)

# Read data
adge_masl = sc.read_h5ad(adge_file)
adata_pred_filt_masl = sc.read_h5ad(ad_sp_file)
# Subsetting adge based on cells in adata_pred (should not have undetermined cells)
subset_cells_masl = adata_pred_filt_masl.obs_names  # Extract cell names from the subset object
# Subset the original AnnData object
adge_filt_masl = adge_masl[adge_masl.obs_names.isin(subset_cells_masl)].copy()

# Subset and sync spatial data
adge_filt_masl.obs['pred_cell_types'] = adata_pred_filt_masl.obs['pred_cell_types']
adge_filt_masl.obsm['spatial'] = adata_pred_filt_masl.obsm['spatial']
adge_filt_masl.uns['spatial'] = adata_pred_filt_masl.uns['spatial']

# Plot spatial data by cell type
#plot_spatial_by_celltype(adata_pred_filt_masl, outdir=patdir, sample_nm=sample_nm, palette_dict=color_map)

# Create expression heatmaps
#create_expression_heatmaps(
#    raw_adata=adata_pred_filt,
#    imputed_adata=adge_filt,
#    markers=gene_markers,
#    groupby="pred_cell_types",
#    outdir=patdir,
#    sample_name=sample_nm
#)

In [15]:
adge_filt_masl

AnnData object with n_obs × n_vars = 270291 × 32528
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'pred_cell_types'
    uns: 'hvg', 'rank_genes_groups', 'spatial'
    obsm: 'spatial'

In [16]:
adata_pred_filt_masl

AnnData object with n_obs × n_vars = 270291 × 17791
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'celltype_proj', 'pred_cell_types'
    uns: 'celltype_proj_colors', 'spatial'
    obsm: 'spatial', 'spatial_cropped', 'tangram_ct_pred'

In [17]:
sc.pp.subsample(adge_filt_masl, n_obs=100000, random_state=42)


In [18]:
adge_filt_masl

AnnData object with n_obs × n_vars = 100000 × 32528
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'pred_cell_types'
    uns: 'hvg', 'rank_genes_groups', 'spatial'
    obsm: 'spatial'

In [ ]:
# READ IN NORMAL

In [19]:
# normal
sample_nm = 'HL160029_filt'

In [20]:
sample_dir = os.path.join(idir, sample_nm)
adge_file = os.path.join(sample_dir, "merge", "adge_merge.h5ad")

filt_h5ad_sample_dir = os.path.join(filt_h5ad_dir, sample_nm)
ad_sp_file = os.path.join(filt_h5ad_sample_dir, "adata_0.5_tangram_filt.h5ad")
patdir = os.path.join(odir, sample_nm)
os.makedirs(patdir, exist_ok=True)

# Read data
adge_normal = sc.read_h5ad(adge_file)
adata_pred_filt_normal = sc.read_h5ad(ad_sp_file)
# Subsetting adge based on cells in adata_pred (should not have undetermined cells)
subset_cells_normal = adata_pred_filt_normal.obs_names  # Extract cell names from the subset object
# Subset the original AnnData object
adge_filt_normal = adge_normal[adge_normal.obs_names.isin(subset_cells_normal)].copy()

# Subset and sync spatial data
adge_filt_normal.obs['pred_cell_types'] = adata_pred_filt_normal.obs['pred_cell_types']
adge_filt_normal.obsm['spatial'] = adata_pred_filt_normal.obsm['spatial']
adge_filt_normal.uns['spatial'] = adata_pred_filt_normal.uns['spatial']


In [21]:
adata_pred_filt_normal

AnnData object with n_obs × n_vars = 248818 × 18029
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'celltype_proj', 'pred_cell_types'
    uns: 'celltype_proj_colors', 'spatial'
    obsm: 'spatial', 'spatial_cropped', 'tangram_ct_pred'

In [22]:
adge_filt_normal

AnnData object with n_obs × n_vars = 248818 × 32507
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'pred_cell_types'
    uns: 'hvg', 'rank_genes_groups', 'spatial'
    obsm: 'spatial'

In [23]:
sc.pp.subsample(adge_filt_normal, n_obs=100000, random_state=42)


In [24]:
adge_filt_normal

AnnData object with n_obs × n_vars = 100000 × 32507
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'pred_cell_types'
    uns: 'hvg', 'rank_genes_groups', 'spatial'
    obsm: 'spatial'

In [16]:
# READ IN MASH

In [25]:
adge_file

'/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/deep_seq/HL160029_filt/merge/adge_merge.h5ad'

In [26]:
# mash
sample_nm = 'HL20221019_broad_no_mast_filt'

In [27]:
idir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/'

In [28]:
sample_dir = os.path.join(idir, sample_nm)
adge_file = os.path.join(sample_dir, "merge", "adge_merge.h5ad")

filt_h5ad_sample_dir = os.path.join(filt_h5ad_dir, sample_nm)
ad_sp_file = os.path.join(filt_h5ad_sample_dir, "adata_0.5_tangram_filt.h5ad")
patdir = os.path.join(odir, sample_nm)
os.makedirs(patdir, exist_ok=True)

# Read data
adge_mash = sc.read_h5ad(adge_file)
adata_pred_filt_mash = sc.read_h5ad(ad_sp_file)
# Subsetting adge based on cells in adata_pred (should not have undetermined cells)
subset_cells_mash = adata_pred_filt_mash.obs_names  # Extract cell names from the subset object
# Subset the original AnnData object
adge_filt_mash = adge_mash[adge_mash.obs_names.isin(subset_cells_mash)].copy()

# Subset and sync spatial data
adge_filt_mash.obs['pred_cell_types'] = adata_pred_filt_mash.obs['pred_cell_types']
adge_filt_mash.obsm['spatial'] = adata_pred_filt_mash.obsm['spatial']
adge_filt_mash.uns['spatial'] = adata_pred_filt_mash.uns['spatial']


In [29]:
adata_pred_filt_mash

AnnData object with n_obs × n_vars = 270311 × 17931
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'clusters', 'uniform_density', 'rna_count_based_density', 'celltype_proj', 'pred_cell_types'
    uns: 'celltype_proj_colors', 'clusters', 'hvg', 'neighbors', 'pca', 'spatial', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial', 'spatial_cropped', 'tangram_ct_pred'

In [30]:
adge_filt_mash

AnnData object with n_obs × n_vars = 270311 × 32681
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'clusters', 'uniform_density', 'rna_count_based_density', 'pred_cell_types'
    uns: 'hvg', 'rank_genes_groups', 'spatial'
    obsm: 'spatial'

In [31]:
sc.pp.subsample(adge_filt_mash, n_obs=100000, random_state=42)


In [32]:
adge_filt_mash

AnnData object with n_obs × n_vars = 100000 × 32681
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'clusters', 'uniform_density', 'rna_count_based_density', 'pred_cell_types'
    uns: 'hvg', 'rank_genes_groups', 'spatial'
    obsm: 'spatial'

In [33]:
adge_filt_masl

AnnData object with n_obs × n_vars = 100000 × 32528
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'pred_cell_types'
    uns: 'hvg', 'rank_genes_groups', 'spatial'
    obsm: 'spatial'

In [34]:
adge_filt_normal

AnnData object with n_obs × n_vars = 100000 × 32507
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'pred_cell_types'
    uns: 'hvg', 'rank_genes_groups', 'spatial'
    obsm: 'spatial'

In [ ]:
# READ IN mash fibstage 2
# HL170058

In [35]:
idir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/deep_seq/'

In [36]:
sample_nm = 'HL170058_filt'

In [37]:
# idir for imputed
# idir for spatial with predicted celltype
sample_dir = os.path.join(idir, sample_nm)
adge_file = os.path.join(sample_dir, "merge", "adge_merge.h5ad")

filt_h5ad_sample_dir = os.path.join(filt_h5ad_dir, sample_nm)
ad_sp_file = os.path.join(filt_h5ad_sample_dir, "adata_0.5_tangram_filt.h5ad")
patdir = os.path.join(odir, sample_nm)
os.makedirs(patdir, exist_ok=True)

# Read data
adge_mash_fibstage2 = sc.read_h5ad(adge_file)
adata_pred_filt_mash_fibstage2 = sc.read_h5ad(ad_sp_file)
# Subsetting adge based on cells in adata_pred (should not have undetermined cells)
subset_cells_mash_fibstage2 = adata_pred_filt_mash_fibstage2.obs_names  # Extract cell names from the subset object
# Subset the original AnnData object
adge_filt_mash_fibstage2 = adge_mash_fibstage2[adge_mash_fibstage2.obs_names.isin(subset_cells_mash_fibstage2)].copy()

# Subset and sync spatial data
adge_filt_mash_fibstage2.obs['pred_cell_types'] = adata_pred_filt_mash_fibstage2.obs['pred_cell_types']
adge_filt_mash_fibstage2.obsm['spatial'] = adata_pred_filt_mash_fibstage2.obsm['spatial']
adge_filt_mash_fibstage2.uns['spatial'] = adata_pred_filt_mash_fibstage2.uns['spatial']

In [38]:
adata_pred_filt_mash_fibstage2

AnnData object with n_obs × n_vars = 315731 × 18035
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'celltype_proj', 'pred_cell_types'
    uns: 'celltype_proj_colors', 'spatial'
    obsm: 'spatial', 'spatial_cropped', 'tangram_ct_pred'

In [39]:
adge_filt_mash_fibstage2

AnnData object with n_obs × n_vars = 315731 × 32681
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'pred_cell_types'
    uns: 'hvg', 'rank_genes_groups', 'spatial'
    obsm: 'spatial'

In [40]:
sc.pp.subsample(adge_filt_mash_fibstage2, n_obs=100000, random_state=42)


In [41]:
adge_filt_mash_fibstage2

AnnData object with n_obs × n_vars = 100000 × 32681
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'pred_cell_types'
    uns: 'hvg', 'rank_genes_groups', 'spatial'
    obsm: 'spatial'

In [42]:
import datetime

In [43]:
print(datetime.datetime.now())


2025-03-21 12:37:42.717035


In [44]:
import anndata as ad

In [51]:
## takes like 10 min to merge everything

In [45]:
print(datetime.datetime.now())

adge_combined = ad.concat(
    [adge_filt_masl, adge_filt_normal, adge_filt_mash, adge_filt_mash_fibstage2 ],
    axis=0,
    join='outer',
    uns_merge='unique',
    label='batch',                # Add a label for batch information
    keys=['HL230324_filt', 'HL160029_filt', 'HL20221019_broad_no_mast_filt', 'HL170058_filt'],  # Batch names
    index_unique='-'
)
print(datetime.datetime.now())


2025-03-21 12:37:48.457077
2025-03-21 12:40:24.989982


In [46]:
import datetime
import anndata as ad

In [47]:
#print(datetime.datetime.now())

#adge_combined = ad.concat(
#    [adge_filt_masl, adge_filt_normal, adge_filt_mash ],
#    axis=0,
#    join='outer',
#    uns_merge='unique',
#    label='batch',                # Add a label for batch information
#    keys=['HL230324_filt', 'HL160029_filt', 'HL20221019_broad_no_mast_filt'],  # Batch names
#    index_unique='-'
#)
#print(datetime.datetime.now())


In [48]:
adge_combined

AnnData object with n_obs × n_vars = 400000 × 34216
    obs: 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'uniform_density', 'rna_count_based_density', 'pred_cell_types', 'clusters', 'batch'
    uns: 'hvg', 'rank_genes_groups', 'spatial'
    obsm: 'spatial'

In [43]:
print("test")

test


In [ ]:
#adge_comb = ad.concat(adata_list_adge, axis=0, uns_merge = 'unique', join = 'outer')


In [49]:
adge_combined.obs['batch'].value_counts()

batch
HL230324_filt                    100000
HL160029_filt                    100000
HL20221019_broad_no_mast_filt    100000
HL170058_filt                    100000
Name: count, dtype: int64

In [50]:
adge_combined.obs['batch'].value_counts()

batch
HL230324_filt                    100000
HL160029_filt                    100000
HL20221019_broad_no_mast_filt    100000
HL170058_filt                    100000
Name: count, dtype: int64

In [51]:

neon_rainbow_colors_broad = {
        'Hepatocytes': '#FF007F',    # Neon pink
        'Myeloid': '#0000FF',           # Neon blue
        'T_NK': '#FFFA00',              # Neon yellow
        'Cholangiocyte': '#00FF00',  # Neon green
        'HSC': '#00FFFF',            # Neon cyan
        'Mast': '#FF1900',        # Neon red
        'Endothelial': '#8A2BE2',    # Neon purple
        'Schwann': '#FF1493',        # Neon deep pink
        'NK': '#FFFDBB',             # lighter yellow
        'B': '#FF8800'               # Neon hot pink
    }

In [ ]:
# removing thy1 and aspn

In [52]:
  brin_markers = [
        
        "cd19", "ms4a1",
        "krt19", "fxyd2", "spp1", 
        "epcam", "sox9", "anxa4",
        #"sry", 
        "krt1",
        #"pecam1",  can't find
        "cd146", 
        "mcam",
        #"sele", 
        "flt4", 
        #"lyve1", 
         # "mcam",
        "cd34", 
       # "ptprc", 
        "stab2" , "ptprb",    # added endothelial
        "col1a1", 
        #"fap",
        "adamts13",
        "ngfr", "cygb", "hgf", "rbp1", #added stellate cells
        #"msln", 
        #"grem1",
      #"calca", 
        "eln", # Gremlin1, Asporin, calcitonin a, Elastin #added fibroblasts
        #"msln", #not found in imputated
        "cyp2e1", "hnf4a", "crp", "alb", 
        "serpina1", "ttr", #added heps
        "c1qa","cd163", "timd4",
        "cd68", "ms4a7", # added myeloid
        "cd69", "trbc2", "cd3d"
    
    
    ]





In [53]:
patdir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/misc/adge_combined4_ds_nash_fibstage2_rm_thy1_aspn/'

In [54]:
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import os

def create_expression_heatmaps(raw_adata, imputed_adata, markers, groupby, outdir, sample_name):
    """
    Generates heatmaps of average gene expression for both raw and imputed AnnData objects.

    Parameters:
    -----------
    raw_adata : AnnData
        AnnData object with raw counts.
    imputed_adata : AnnData
        AnnData object with imputed counts.
    markers : list
        List of marker genes to include in the heatmap.
    groupby : str
        Column in `obs` to group cells by (e.g., cell types).
    outdir : str
        Directory to save the output plots.
    sample_name : str
        Name of the sample for labeling files and plots.

    Returns:
    --------
    None
    """
    os.makedirs(outdir, exist_ok=True)  # Create output directory if it doesn't exist


    # Function to create heatmap from AnnData object
    def plot_heatmap(adata, dataset_type, markers):
        markers = [gene for gene in markers if gene in adata.var_names]
        
        # Generate dot plot
        dotplot = sc.pl.dotplot(
            adata,
            var_names=markers,
            groupby=groupby,
            color_map="viridis",
            dot_max=1,
            dot_min=0,
            vmin=0.2,
            expression_cutoff=0.1,
            show=False,
            standard_scale="var",
            return_fig=True
        )

        # Extract average expression data
        avg_expression = dotplot.dot_color_df

        # Create heatmap
        plt.figure(figsize=(14, 20))
        col_map_red = mpl.colormaps['Reds'].resampled(256)
        sns.heatmap(
            avg_expression.T,
            #cmap="viridis",
            cmap=col_map_red,
            fmt=".2f",
            linewidths=0.5,
            cbar_kws={"label": "Average Expression"}
        )
        plt.xlabel("Genes")
        plt.ylabel("Cell Groups")
        plt.title(f"Heatmap of Average Gene Expression ({dataset_type} Counts) - {sample_name}")
        plt.tight_layout()

        # Save the heatmap
        heatmap_filename = os.path.join(outdir, f"{sample_name}_{dataset_type}_heatmap.pdf")
        plt.savefig(heatmap_filename)
        plt.close()

    # Generate heatmap for raw counts
    #plot_heatmap(raw_adata, "raw", markers)

    # Generate heatmap for imputed counts
    plot_heatmap(imputed_adata, "imputed", markers)




In [55]:
patdir

'/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/misc/adge_combined4_ds_nash_fibstage2_rm_thy1_aspn/'

In [59]:
sample_nm

'HL170058_filt'

In [56]:
sample_nm = 'combined_normal_masl_mash_fibstage2_rm_thy1_aspn'

In [57]:
#brin_markers

In [58]:
create_expression_heatmaps(
    raw_adata=adata_pred_filt_masl, #just as a filler i don't use this
    imputed_adata=adge_combined,
    markers=brin_markers,
    groupby="pred_cell_types",
    outdir=patdir,
    sample_name=sample_nm
)


In [ ]:
# QY_2673_1_2	HL230324	metald,   80G
# QY_2665_1_2	HL160029	Normal,  90G
# QY_2480_Liver_MASH	HL2201019	MASH HL20221019_broad_no_mast_filt/merge/adge_merge.h5ad (88GB)

# HL170058_filt MASH QY_2666

In [ ]:
# Create expression heatmaps
#create_expression_heatmaps(
#    raw_adata=adata_pred_filt,
#    imputed_adata=adge_filt,
#    markers=gene_markers,
#    groupby="pred_cell_types",
#    outdir=patdir,
#    sample_name=sample_nm
#)